In [1]:
import pandas as pd
import datetime
import chess
from utils import display_fen, get_db_connection
import matplotlib.pyplot as plt
import numpy as np
%load_ext autoreload
%autoreload 2

# create a personal database 
conn = get_db_connection(threads = 10)

In [ ]:
import datetime

# Attach the primary database
try:
    conn.execute("ATTACH '/scratch/gpfs/GRIFFITHS/chess-db/lichess.db' AS core (READ_ONLY)")
except Exception as e:
    print(f"warning: {e}")

# Get games table
start_date = "2023-12-01"
end_date = "2023-12-02"

conn.sql(f"""CREATE OR REPLACE TABLE games AS SELECT * FROM core.games WHERE utc_datetime BETWEEN '{start_date}' AND '{end_date}'""")
games = conn.sql("""SELECT * FROM games""").df()

conn.sql("""
    CREATE OR REPLACE TABLE endgame_positions AS
    SELECT * FROM core.moves m
    JOIN games g ON m.gid = g.gid
    WHERE length(regexp_replace(m.board_position, '[^a-zA-Z]', '', 'g')) BETWEEN 6 AND 8
    AND g.initial_clock >= 300
    AND g.white_elo >= 2000
    AND g.black_elo >= 2000;
;""")

endgame_positions = conn.sql("""SELECT * FROM endgame_positions""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [11]:
from sklearn.model_selection import train_test_split

# sample and split

sampled = endgame_positions[endgame_positions.game_end_type == "Normal"].sample(n=10000, random_state=42)
fit_positions, eval_positions = train_test_split(sampled, test_size=0.5, random_state=42)

In [ ]:
# how much better can we do? better model of VOC?
# architecture includes in it multiple steps - network has a decision to keep thinking or not
# if that model already explains more variance in RT than Evan's model
# each iteration is one-step lookahead vs each iteration is a full rollout



{'Normal'}

In [ ]:
def active_player_won(row): 
    if row.player_white and row.white_won: return 1
    if not row.player_white and row.black_won: return 1
    if not row.white_won and not row.black_won: return 0.5
    return 0

fit_positions["active_player_won"] = fit_positions.apply(active_player_won, axis=1)


In [ ]:
endgame_positions.keys

0          True
1          True
2          True
3          True
4          True
          ...  
188749    False
188750    False
188751    False
188752    False
188753    False
Name: black_won, Length: 188754, dtype: bool

In [ ]:
from chess.engine import EngineTerminatedError
from tqdm import tqdm
from utils import get_stockfish_engine
import numpy as np

SHALLOW_DEPTH = 1
DEEP_DEPTH = 4

engine = get_stockfish_engine()

data = []
for i, position in tqdm(enumerate(endgame_positions.itertuples())):
    if i > 200:
        break

    board = chess.Board(position.board_position)
    try:
        player = 1 if board.turn == chess.WHITE else -1
        candidate_moves = set()
        shallow_move = None

        # clear the hash here so we get a fresh analysis each time
        engine.configure({"Clear Hash": True})
        with engine.analysis(board, chess.engine.Limit(depth=DEEP_DEPTH)) as analysis:
            for info in analysis:
                depth = info.get("depth")
                pv = info.get("pv")
                if pv:
                    if depth == SHALLOW_DEPTH:
                        shallow_move = pv[0]
                    candidate_moves.add(pv[0])

        if shallow_move is None or len(candidate_moves) == 0:
            continue
    
        engine.configure({"Clear Hash": True})
        infos = engine.analyse(
            board,
            chess.engine.Limit(depth=DEEP_DEPTH),
            multipv=len(candidate_moves),
            root_moves=list(candidate_moves)
        )
        scores = {
            info["pv"][0]: info["score"].white().score(mate_score=10000) * player
            for info in infos
        }

        if shallow_move not in scores:
            continue

        v_shallow = scores[shallow_move]
        v_deep = max(scores.values())
        voc = v_deep - v_shallow

        data.append(pd.Series({
            "board_position": position.board_position,
            "player_white": position.player_white,
            "score_shallow": v_shallow,
            "score_deep": v_deep,
            "voc": voc,
            "move_time": position.move_time,
        }))

    except EngineTerminatedError:
        print("Engine crashed on position", position.board_position)
        try:
            engine.quit()
        except Exception:
            pass
        engine = get_stockfish_engine()

engine.quit()

In [ ]:
df = pd.DataFrame(data)


In [ ]:
plt.hist(df.score_deep - df.score_shallow)
# plt.xlim(-30, 30)
# plt.ylim(-30, 30)

In [ ]:
df = pd.DataFrame(data)

import matplotlib.pyplot as plt
plt.scatter(np.log1p(df.voc), np.log1p(df.move_time), alpha = 0.3)
correlation = df["voc"].corr(df["move_time"])
print("Correlation between VOC and move_time:", correlation)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(games.n_moves, bins = 30)
ax.set(title = "Number of moves in games", xlabel = "Number of moves", ylabel = "Number of games")

fig, ax = plt.subplots(figsize=(4, 5))
games.opening.value_counts().sort_values()[lambda x: x > 80].plot(kind="barh", ax=ax)